# CS 696 · Compare accuracy, coverage and cost on changed queues

Compare two existing reporting policies on the same what-if questions. Count incorrect answers shown, correct answers withheld and work performed. Defend a conclusion whose limits you can state.

Module 05 showed how ordinary replay can conceal an error under a changed scan duration. Bring that distinction into evaluation. A museum archive has eight candidate answers and the response logs of two existing reporting policies. Compare the logs with independently worked schedules to decide what the archive can establish.

In this fictional archive, both policies receive the same candidate finish minute. The size policy reports answers for queues of at most two objects. The complete-input policy reports answers when the archived scan settings are available for every object. Empty means the policy withheld the answer. Separate field cards give the evaluator all eight reference times. Neither reporting policy had access to those cards. H uses three-minute scans, D uses six-minute scans, P compares a regular run with a middle delay, and T uses four-minute scans. Each inspection takes four minutes and all objects arrive at minute zero.

## Count the cost of holding back an answer

Start with the size policy’s archived output. A blank means an answer was withheld, even though the simulator produced a candidate finish time. Score each offered question with the independent field card, including the blanks. A policy’s reported accuracy describes only the answers it showed.

coverage = A / N     error among released = W / A     useful answers withheld = C − K

N counts all questions. A counts released answers. W counts incorrect released answers. C counts all correct candidate answers in the ledger and K counts correct answers released. If A is zero, conditional error is undefined. Here usefulness means giving the correct requested finish minute.

In [1]:
# ID, objects, hand-checked final minute, candidate final minute,
# size-policy output, complete-input-policy output (None = withheld).
rows = [
    ("H1", 1,  7,  7,  7,  7), ("H2", 2, 11, 11, 11, 11),
    ("D1", 2, 16, 14, 14, None), ("D2", 3, 22, 18, None, None),
    ("P1", 3, 15, 15, None, 15), ("P2", 3, 18, 18, None, None),
    ("T1", 2, 12, 11, 11, 11), ("T2", 3, 16, 16, None, None),
]

shown = [(key, answer == ref) for key, n, ref, guess, answer, complete in rows
         if answer is not None]
correct = sum(guess == ref for key, n, ref, guess, size, complete in rows)
kept = sum(match for key, match in shown)
print("shown", [key for key, match in shown], "coverage", len(shown), "/", len(rows))
print("wrong shown", len(shown) - kept, "correct withheld", correct - kept)


shown ['H1', 'H2', 'D1', 'T1'] coverage 4 / 8
wrong shown 2 correct withheld 3


**Discuss** · Could a planner infer the fraction of correct answers withheld from the error among released answers alone?

The size check releases four of eight answers. Two of the four are wrong, but the withheld set contains three of the five correct candidate answers. To recover that latter count you need reference answers for withheld questions as well. Evaluating just what a planner sees hides the cost of withholding useful results.

## Compare policies at the same coverage

The complete-input policy withheld D1, D2, P2 and T2 because their archived scan settings were missing. Its other four answers are the saved candidate minutes. The evaluator can reconstruct reference times even on those withheld cases using separate field cards. Both policies happened to answer four questions. Compare exactly which answers differed between them.

Wⱼ = Σᵢ 𝟙[policy j answered i and its answer differs from referenceᵢ]

𝟙 is one when its condition holds and zero otherwise. j chooses the reporting policy and i indexes the same eight questions in each log. Equal coverage does not require that the two policies answered the same questions.

In [2]:
# ID, objects, hand-checked final minute, candidate final minute,
# size-policy output, complete-input-policy output (None = withheld).
rows = [
    ("H1", 1,  7,  7,  7,  7), ("H2", 2, 11, 11, 11, 11),
    ("D1", 2, 16, 14, 14, None), ("D2", 3, 22, 18, None, None),
    ("P1", 3, 15, 15, None, 15), ("P2", 3, 18, 18, None, None),
    ("T1", 2, 12, 11, 11, 11), ("T2", 3, 16, 16, None, None),
]

for name, column in [("size", 4), ("complete input", 5)]:
    shown = [r for r in rows if r[column] is not None]
    wrong = [r[0] for r in shown if r[column] != r[2]]
    withheld_correct = [r[0] for r in rows if r[column] is None and r[2] == r[3]]
    print(name, "shown", [r[0] for r in shown], "wrong", wrong,
          "correct held", withheld_correct, "accuracy", len(shown) - len(wrong), "/", len(shown))


size shown ['H1', 'H2', 'D1', 'T1'] wrong ['D1', 'T1'] correct held ['P1', 'P2', 'T2'] accuracy 2 / 4
complete input shown ['H1', 'H2', 'P1', 'T1'] wrong ['T1'] correct held ['P2', 'T2'] accuracy 3 / 4


**Discuss** · Why did the complete-input policy avoid D1 but still show the wrong answer to T1? What does the matched comparison actually tell you?

It happened to withhold D1, whose archived settings were missing, and to show correct P1. Accuracy is 3/4 compared with 2/4 at the same coverage of 4/8. T1 had complete inputs, but the candidate minute is 11 where the independently checked answer is 12. Input completeness alone cannot establish a correct calculation.

## Ask what depends on one scenario family

H1 and H2 are variants of one ordinary queue. D1 and D2 share the same long-scan change. Treating each variant as a new independent scenario exaggerates the diversity of the evaluation. Remove each family in turn and recompute the difference in wrong answers shown per question offered.

gᵢ = 𝟙[size showed a wrong answer] − 𝟙[complete input showed a wrong answer]     G = Σᵢ gᵢ / N

A positive g favors the complete-input policy on that question. N is the number of questions offered to both policies. Including all offered questions keeps this measure defined if a policy answers none. Report coverage alongside it when comparing different question sets.

In [3]:
# ID, objects, hand-checked final minute, candidate final minute,
# size-policy output, complete-input-policy output (None = withheld).
rows = [
    ("H1", 1,  7,  7,  7,  7), ("H2", 2, 11, 11, 11, 11),
    ("D1", 2, 16, 14, 14, None), ("D2", 3, 22, 18, None, None),
    ("P1", 3, 15, 15, None, 15), ("P2", 3, 18, 18, None, None),
    ("T1", 2, 12, 11, 11, 11), ("T2", 3, 16, 16, None, None),
]

for omit in (None, "H", "D", "P", "T"):
    kept = [r for r in rows if r[0][0] != omit]
    difference = 0
    for key, n, ref, guess, size, complete in kept:
        difference += int(size is not None and size != ref)
        difference -= int(complete is not None and complete != ref)
    print("omitted family", omit, "fewer wrong answers", difference, "/", len(kept))


omitted family None fewer wrong answers 1 / 8
omitted family H fewer wrong answers 1 / 6
omitted family D fewer wrong answers 0 / 6
omitted family P fewer wrong answers 1 / 6
omitted family T fewer wrong answers 1 / 6


**Discuss** · Can these eight observations establish that the complete-input policy improves answers on new kinds of changed queues?

The observed reduction is one wrong answer per eight offered questions. Removing D gives zero per six. Removing H, P or T gives one per six. All of the observed reduction comes from D1. These four omission results describe the sensitivity of the conclusion to the particular families in the archive.

## Independent task

The archive has a work log. The size policy used one unit per question. The complete-input policy used one lookup per question plus one unit per object when the scan settings were available. Compute total work, wrong answers shown and correct answers withheld. Which existing policy would you choose if a wrong answer costs six, a correct answer withheld costs one, and a work unit costs either 0.5 or 1? Find the break-even work-unit price.

In [4]:
# ID, objects, hand-checked final minute, candidate final minute,
# size-policy output, complete-input-policy output (None = withheld).
rows = [
    ("H1", 1,  7,  7,  7,  7), ("H2", 2, 11, 11, 11, 11),
    ("D1", 2, 16, 14, 14, None), ("D2", 3, 22, 18, None, None),
    ("P1", 3, 15, 15, None, 15), ("P2", 3, 18, 18, None, None),
    ("T1", 2, 12, 11, 11, 11), ("T2", 3, 16, 16, None, None),
]

for key, n, ref, guess, size, complete in rows:
    complete_work = 1 + (n if complete is not None else 0)
    print(key, "size work", 1, "complete-input work", complete_work)


H1 size work 1 complete-input work 2
H2 size work 1 complete-input work 3
D1 size work 1 complete-input work 1
D2 size work 1 complete-input work 1
P1 size work 1 complete-input work 4
P2 size work 1 complete-input work 1
T1 size work 1 complete-input work 3
T2 size work 1 complete-input work 1


## Compare your solution

For a policy with W wrong answers shown, H correct answers withheld and T work units, total loss is 6W + H + pT. Compute each term before combining them. Prices and work counts describe this invented exercise.

In [5]:
# ID, objects, hand-checked final minute, candidate final minute,
# size-policy output, complete-input-policy output (None = withheld).
rows = [
    ("H1", 1,  7,  7,  7,  7), ("H2", 2, 11, 11, 11, 11),
    ("D1", 2, 16, 14, 14, None), ("D2", 3, 22, 18, None, None),
    ("P1", 3, 15, 15, None, 15), ("P2", 3, 18, 18, None, None),
    ("T1", 2, 12, 11, 11, 11), ("T2", 3, 16, 16, None, None),
]

stats = {}
for name, column in [("size", 4), ("complete input", 5)]:
    wrong = sum(r[column] is not None and r[column] != r[2] for r in rows)
    held = sum(r[column] is None and r[3] == r[2] for r in rows)
    work = len(rows)
    if name == "complete input":
        work += sum(r[1] for r in rows if r[5] is not None)
    stats[name] = (wrong, held, work)
    for price in (0.5, 1):
        print(name, "wrong/held/work", (wrong, held, work),
              "price", price, "loss", 6 * wrong + held + price * work)
sw, sh, st = stats["size"]
cw, ch, ct = stats["complete input"]
print("break-even work price", (6 * (sw - cw) + sh - ch) / (ct - st))


size wrong/held/work (2, 3, 8) price 0.5 loss 19.0
size wrong/held/work (2, 3, 8) price 1 loss 23
complete input wrong/held/work (1, 2, 16) price 0.5 loss 16.0
complete input wrong/held/work (1, 2, 16) price 1 loss 24
break-even work price 0.875


The size policy uses eight units, shows two wrong answers and withholds three correct ones. The complete-input policy uses 16 units, shows one wrong answer and withholds two correct ones. At price 0.5 the losses are 19 and 16, favoring complete inputs. At price 1 they are 23 and 24, favoring size. They tie at 7/8 per work unit. State the choice conditionally on this archive and these assumed costs. A claim about actual staff time would require timing data.

## Mentor review

Bring both archived response logs, coverage and accuracy with denominators, work calculations and a short recommendation. Identify a changed queue whose independent record would give you evidence beyond these four families.

- Count all eight questions and all correct answers withheld, including cases with missing archived inputs.
- Separate the evidence used for scoring from information available to each reporting policy. Identify one question answered only by each policy.
- Report both prices, name the family driving the observed difference, and state the limits of the recommendation.
- If nobody reconstructed the withheld answers, which of your claims could still be checked?
- Would missing inputs on slow-scanner days affect which cases appear in the complete-input log?
- Which original record would let a second reader audit the claim about T1 without trusting the candidate?

Carry a claim from the comparison, the specific supporting records, the alternative explanation and the evidence still needed into Module 07. There you will check what a new set of records actually establishes. Keep each claim attached to the observation or rule that could support or overturn it.